In [2]:
import os
print(os.getcwd())

C:\Users\User\Documents\GitHub\transport-analytics\notebooks


In [3]:
import pdfplumber

pdf_path = "../data/raw/Stat2025-EN.pdf"

with pdfplumber.open(pdf_path) as pdf:
    print(f"Total pages in PDF: {len(pdf.pages)}\n")

    search_terms = ["Table 5.8", "Table 5.1", "Table 6.2", "Table 6.5"]

    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        for term in search_terms:
            if term in text:
                print(f"pdfplumber index {i}  ->  found '{term}'")

Total pages in PDF: 261

pdfplumber index 11  ->  found 'Table 5.8'
pdfplumber index 11  ->  found 'Table 5.1'
pdfplumber index 11  ->  found 'Table 6.2'
pdfplumber index 11  ->  found 'Table 6.5'
pdfplumber index 65  ->  found 'Table 5.1'
pdfplumber index 76  ->  found 'Table 5.8'
pdfplumber index 79  ->  found 'Table 6.2'
pdfplumber index 81  ->  found 'Table 6.5'


In [4]:
import pdfplumber

pdf_path = "../data/raw/Stat2025-EN.pdf"

with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        # "Fleet (Average)" is a row label that only exists in the real
        # Table 5.8 data — it won't appear in a ToC line, which is just
        # "Table 5.8 Operational Data - SLTB    56"
        if "Fleet (Average)" in text and "Operational Data" in text:
            print(f"pdfplumber index {i}  ->  looks like the real Table 5.8")
            print("---- text preview ----")
            print(text[:600])
            print("-----------------------\n")

pdfplumber index 76  ->  looks like the real Table 5.8
---- text preview ----
Figure 5.16 Fuel Efficiency – SLTB
Table 5.7 Financial Progress
Item 2018 2019 2020 2021 2022 2023 2024
Total Revenue (Rs.Mn.) 44,102.92 43,489.53 31,233.25 26,963.77 71,801.06 77,768.79 100,942.83
Total Cost (Rs.Mn.) 39,956.73 39,954.65 31,586.35 28,384.75 71,356.58 74,976.83 95,348.34
Profit/Loss 4,146.19 3,534.87 353.10 1,420.98 444.48 2,791.95 5594. 48
(Rs.Mn.)
Source: Sri Lanka Transport Board
Table 5.8 Operational Data
Operational Data 2018 2019 2020 2021 2022 2023 2024
Fleet (Average) 7,125 7,012 7,118 7,034 6,865 7,218 7,135
T.T.R (Average) (Time 7,257 7,284 7,338 7,288 7,058 7,204 7,3
-----------------------



In [1]:
import pdfplumber

pdf_path = "../data/raw/Stat2025-EN.pdf"
TARGET_PAGE = 76

table_settings = {
    "vertical_strategy": "text",
    "horizontal_strategy": "text",
}

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[TARGET_PAGE]
    tables = page.extract_tables(table_settings)

print(f"Number of tables detected with 'text' strategy: {len(tables)}\n")
for t_idx, table in enumerate(tables):
    print(f"=== Table {t_idx} — {len(table)} rows ===")
    for row in table:
        print(row)
    print()

Number of tables detected with 'text' strategy: 1

=== Table 0 — 42 rows ===
['', '', '', '', '', '', 'Figure 5.1', '6', 'Fuel Efficien', 'cy – SLTB', '', '', '', '', '', '']
['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
['', '', '', '', '', '', 'Table 5.', '7', 'Financial', 'Progress', '', '', '', '', '', '']
['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
['', 'Item', '', '', '2018', '', '2019', '', '2020', '2021 20', '22', '20', '2', '3', '20', '24']
['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
['', 'Total Revenue', '(Rs.M', 'n.)', '44,102.9', '2', '43,489.53', '', '31,233.25 2', '6,963.77 71,8', '01.06', '77,76', '8', '.79', '100,9', '42.83']
['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
['', 'Total Cost (Rs', '.Mn.)', '', '39,956.7', '3', '39,954.65', '', '31,586.35 2', '8,384.75 71,3', '56.58', '74,97', '6', '.83', '95,3', '48.34']
['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
['', 'Prof

In [2]:
raw_rows = tables[0]  # the one big table from the 'text' strategy

joined_rows = []
for row in raw_rows:
    cells = [c.strip() for c in row if c and c.strip()]
    if cells:
        joined_rows.append(" ".join(cells))

for i, r in enumerate(joined_rows):
    print(i, "->", r)

0 -> Figure 5.1 6 Fuel Efficien cy – SLTB
1 -> Table 5. 7 Financial Progress
2 -> Item 2018 2019 2020 2021 20 22 20 2 3 20 24
3 -> Total Revenue (Rs.M n.) 44,102.9 2 43,489.53 31,233.25 2 6,963.77 71,8 01.06 77,76 8 .79 100,9 42.83
4 -> Total Cost (Rs .Mn.) 39,956.7 3 39,954.65 31,586.35 2 8,384.75 71,3 56.58 74,97 6 .83 95,3 48.34
5 -> Profit/Loss 4,146.1 9 3,534.87 353.10 1 ,420.98 44 4.48 2,79 1 .95 559 4. 48
6 -> (Rs.Mn.)
7 -> Source : S ri Lanka Transpo rt Board
8 -> Table 5 .8 Operatio nal Data
9 -> Op erational Data 2018 201 9 2 020 2021 2022 2 023 2024
10 -> Flee t (Average) 7,125 7,012 7, 118 7,034 6,865 7 ,218 7,135
11 -> T.T. R (Average) (Time 7,257 7,284 7, 338 7,288 7,058 7 ,204 7,307
12 -> Tab le Requirement)
13 -> Ave rage Buses Made 5,970 5,854 5,6 41 5,181 4,814 5 ,243 5,107
14 -> Ava ilable a Day
15 -> Ave rage Buses Oper ated a 5,227 5,079 3,9 71 4,449 4,279 4 ,379 4,581
16 -> Day
17 -> Ave rage Scheduled K m 1,687,452 1,682 ,087 1,7 02,058 1,648,704 1,777,719 2 56
1

In [3]:
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[TARGET_PAGE]
    text = page.extract_text()

lines = text.split("\n")
for i, line in enumerate(lines):
    print(i, "->", line)

0 -> Figure 5.16 Fuel Efficiency – SLTB
1 -> Table 5.7 Financial Progress
2 -> Item 2018 2019 2020 2021 2022 2023 2024
3 -> Total Revenue (Rs.Mn.) 44,102.92 43,489.53 31,233.25 26,963.77 71,801.06 77,768.79 100,942.83
4 -> Total Cost (Rs.Mn.) 39,956.73 39,954.65 31,586.35 28,384.75 71,356.58 74,976.83 95,348.34
5 -> Profit/Loss 4,146.19 3,534.87 353.10 1,420.98 444.48 2,791.95 5594. 48
6 -> (Rs.Mn.)
7 -> Source: Sri Lanka Transport Board
8 -> Table 5.8 Operational Data
9 -> Operational Data 2018 2019 2020 2021 2022 2023 2024
10 -> Fleet (Average) 7,125 7,012 7,118 7,034 6,865 7,218 7,135
11 -> T.T.R (Average) (Time 7,257 7,284 7,338 7,288 7,058 7,204 7,307
12 -> Table Requirement)
13 -> Average Buses Made 5,970 5,854 5,641 5,181 4,814 5,243 5,107
14 -> Available a Day
15 -> Average Buses Operated a 5,227 5,079 3,971 4,449 4,279 4,379 4,581
16 -> Day
17 -> Average Scheduled Km 1,687,452 1,682,087 1,702,058 1,648,704 1,777,719 256
18 -> Scheduled AVU KMs
19 -> Operated kilometers 446,287

In [4]:
import re

def extract_numbers(line):
    # Matches things like 1,234,567.89 / 353.10 / -353,102,715
    return re.findall(r'-?\d[\d,]*\.?\d*', line)

def merge_split_decimals(numbers):
    # Fixes a specific PDF quirk: "5594." and "48" printed as two
    # separate tokens due to a rendering gap — should be "5594.48"
    merged, skip = [], False
    for idx, num in enumerate(numbers):
        if skip:
            skip = False
            continue
        if num.endswith('.') and idx + 1 < len(numbers) and re.fullmatch(r'\d{1,2}', numbers[idx + 1]):
            merged.append(num + numbers[idx + 1])
            skip = True
        else:
            merged.append(num)
    return merged

def is_furniture(line):
    # Page furniture we don't want as data rows
    keywords = ["Figure", "Table 5.", "Source:", "Item ", "Operational Data 20"]
    return any(k in line for k in keywords) or line.strip().isdigit()

lines = text.split("\n")
rows = []
i = 0
while i < len(lines):
    line = lines[i].strip()
    if not line or is_furniture(line):
        i += 1
        continue

    numbers = extract_numbers(line)
    if numbers:
        numbers = merge_split_decimals(numbers)
        first_num_start = re.search(r'-?\d[\d,]*\.?\d*', line).start()
        label = line[:first_num_start].strip()

        # If the NEXT line is label text with no numbers, it's a
        # wrapped continuation of this row's label — merge it in.
        if i + 1 < len(lines):
            nxt = lines[i + 1].strip()
            if nxt and not extract_numbers(nxt) and not is_furniture(nxt):
                label = f"{label} {nxt}"
                i += 1

        rows.append({"label": label, "values": numbers, "n_values": len(numbers)})
    i += 1

for r in rows:
    print(r)

{'label': 'Total Revenue (Rs.Mn.)', 'values': ['44,102.92', '43,489.53', '31,233.25', '26,963.77', '71,801.06', '77,768.79', '100,942.83'], 'n_values': 7}
{'label': 'Total Cost (Rs.Mn.)', 'values': ['39,956.73', '39,954.65', '31,586.35', '28,384.75', '71,356.58', '74,976.83', '95,348.34'], 'n_values': 7}
{'label': 'Profit/Loss (Rs.Mn.)', 'values': ['4,146.19', '3,534.87', '353.10', '1,420.98', '444.48', '2,791.95', '5594.48'], 'n_values': 7}
{'label': 'Fleet (Average)', 'values': ['7,125', '7,012', '7,118', '7,034', '6,865', '7,218', '7,135'], 'n_values': 7}
{'label': 'T.T.R (Average) (Time Table Requirement)', 'values': ['7,257', '7,284', '7,338', '7,288', '7,058', '7,204', '7,307'], 'n_values': 7}
{'label': 'Average Buses Made Available a Day', 'values': ['5,970', '5,854', '5,641', '5,181', '4,814', '5,243', '5,107'], 'n_values': 7}
{'label': 'Average Buses Operated a Day', 'values': ['5,227', '5,079', '3,971', '4,449', '4,279', '4,379', '4,581'], 'n_values': 7}
{'label': 'Average Sc

In [5]:
import pandas as pd

YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024]

records = []
for r in rows:
    values = r["values"]
    # pad short rows (e.g. the 6-value Scheduled Km row) with None
    # for the missing trailing year, rather than guessing a number
    if len(values) < len(YEARS):
        values = values + [None] * (len(YEARS) - len(values))

    for year, val in zip(YEARS, values):
        clean_val = None if val is None else float(val.replace(",", ""))
        records.append({"metric": r["label"], "year": year, "value": clean_val})

df = pd.DataFrame(records)
df

,metric,year,value
0,Total Revenue (Rs.Mn.),2018,4.410292e+04
1,Total Revenue (Rs.Mn.),2019,4.348953e+04
2,Total Revenue (Rs.Mn.),2020,3.123325e+04
3,Total Revenue (Rs.Mn.),2021,2.696377e+04
4,Total Revenue (Rs.Mn.),2022,7.180106e+04
...,...,...,...
100,Profit/Loss*(Rs.),2020,-3.531027e+08
101,Profit/Loss*(Rs.),2021,-1.420985e+09
102,Profit/Loss*(Rs.),2022,4.444805e+08
103,Profit/Loss*(Rs.),2023,2.791956e+09


In [6]:
import os

os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/sltb_financial_operational_2018_2024.csv"
df.to_csv(output_path, index=False)

print(f"Saved {len(df)} rows to {output_path}")

Saved 105 rows to ../data/processed/sltb_financial_operational_2018_2024.csv
